# 在 Colab 上训练 ACT（图像输入）

输入 `observation.state`（6 个关节角）+ `observation.images.overview` + `observation.images.wrist_cam`（240×320 RGB），**不含方块坐标**（`observation.environment_state`）。状态版见 `train_act_colab.ipynb`。

数据集是在本机从录制的状态离线渲染的（`scripts/npz_to_lerobot.py --cameras overview wrist_cam --no-env-state`，渲染需要 mujoco，约 2 小时），所以 Colab 上不再转换，直接解压现成的 LeRobot 数据集：

| 数据集 | 回合 | 帧 | 大小 |
|---|---|---|---|
| `so101_sim_pick_place_images_50` | 前 50 回合 | 25,300 | 156 MB |
| `so101_sim_pick_place_images_291` | 偏移 200 + 重抓 91 | 142,605 | 880 MB |
| `so101_sim_pick_place_images_all_hold` | 正常 200 + 重抓 94 + 边界 100 + 边界重抓 86（末尾 2 s 保持） | 274,790 | 1.6 GB |

**准备**：在 Google Drive 的 `so101_act/` 下上传 `so101_images_datasets.tar`（本机 `data/lerobot/` 下，打包命令：
`tar -C data/lerobot -cf data/lerobot/so101_images_datasets.tar so101_sim_pick_place_images_50 so101_sim_pick_place_images_291`）。480 回合的数据集单独打包成 `so101_images_all_hold.tar`（`tar -C data/lerobot -cf data/lerobot/so101_images_all_hold.tar so101_sim_pick_place_images_all_hold`），也上传到同一目录；解压单元会解压目录里有的所有 tar。

训练输出写到 Drive（`so101_act/outputs/<run>`），断线后可以用最后一个单元续训。运行时类型选 GPU（A100 最好，L4 也可以）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/so101_act'
%cd $PROJECT
!ls -la

In [ ]:
!pip install -q 'lerobot[dataset,training]==0.6.1'
# Colab's preinstalled diffusers needs a newer torchao than the one it ships with; lerobot
# imports diffusers (through GR00T) when it lists its policies and then fails with
# "cannot import name 'FqnToConfig'". ACT needs neither, and lerobot runs without them.
!pip uninstall -y -q diffusers torchao
!python -c "import lerobot, torch; print(lerobot.__version__, torch.__version__, torch.cuda.get_device_name())"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
print('CPU cores:', os.cpu_count(), '(video decoding runs in the dataloader workers: 2 cores is too few)')

## W&B 登录

训练曲线记录到 Weights & Biases 的 `so101-act` 项目（与状态版同一个项目，方便对比）。先在 Colab 左侧栏的 🔑 Secrets 中添加 `WANDB_API_KEY`，并允许本 notebook 访问。不想用 W&B 时把 `USE_WANDB` 改成 `False`。

In [ ]:
import os
USE_WANDB = True
WANDB_PROJECT = 'so101-act'

if USE_WANDB:
    from google.colab import userdata
    try:
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    except userdata.SecretNotFoundError:
        pass
    import wandb
    wandb.login()

## 1. 解压数据集

解压到本地盘 `/content/datasets`：训练时每步都要解码视频，从 Drive 直接读太慢。只需要几十秒。

In [ ]:
!mkdir -p /content/datasets
import glob
for tar in sorted(glob.glob(f'{PROJECT}/so101_images_*.tar')):
    print('extracting', tar)
    get_ipython().system(f'tar -xf {tar} -C /content/datasets')
!du -sh /content/datasets/*

## 2. 训练

| 参数 | 值 | 说明 |
|---|---|---|
| `chunk_size` / `n_action_steps` | 50 / 50 | 与状态版相同（1 s 的动作） |
| `batch_size` | 64（先由吞吐测试确认显存够） | 与状态版相同；显存不够时改 32 |
| `steps` | 60k | 291 回合约 27 个 epoch；每 10k 步存一个 checkpoint，用闭环成功率挑选 |
| `normalization_mapping` | 四类都写 `MEAN_STD` | 数据集里没有 ENV，写上无妨；图像按 ImageNet 统计量归一化（`dataset.use_imagenet_stats` 默认开启） |
| 视觉骨干 | ResNet18，ImageNet 预训练（ACT 默认） | 每个相机单独过骨干，特征拼成 transformer 的输入 token |
| `num_workers` | CPU 核数 | 视频解码在 dataloader 进程里做，worker 少了 GPU 会等数据 |
| 图像增强 | 默认关闭（`IMAGE_AUG`） | 可选：LeRobot 自带的亮度 / 对比度 / 饱和度 / 色调 / 锐度随机变换 |
| 学习率等 | LeRobot 默认（lr 1e-5，backbone lr 1e-5，KL 权重 10，dropout 0.1） | |

### 2.1 吞吐测试

先在 50 回合数据集上跑 300 步（输出写到 `/content/smoke`，不占 Drive）。看日志里的 `updt_s`（每步计算时间）和 `data_s`（等数据的时间）：`data_s` 明显大于 0 说明数据加载是瓶颈。60k 步的耗时 ≈ 60000 × (updt_s + data_s)。

In [ ]:
import json
import shlex

STEPS = 60_000
SAVE_FREQ = 10_000
BATCH_SIZE = 64
NUM_WORKERS = os.cpu_count()
IMAGE_AUG = False
# The flag replaces ACT's whole default mapping, so list every feature type.
NORMALIZATION = {'VISUAL': 'MEAN_STD', 'STATE': 'MEAN_STD', 'ENV': 'MEAN_STD', 'ACTION': 'MEAN_STD'}

def train(run, dataset, steps=STEPS, save_freq=SAVE_FREQ, batch_size=BATCH_SIZE,
          output_root=f'{PROJECT}/outputs', wandb=USE_WANDB, log_freq=500):
    cmd = (
        'lerobot-train'
        ' --policy.type=act'
        ' --policy.chunk_size=50'
        ' --policy.n_action_steps=50'
        ' --policy.device=cuda'
        ' --policy.push_to_hub=false'
        f' --policy.normalization_mapping={shlex.quote(json.dumps(NORMALIZATION))}'
        f' --dataset.repo_id=local/{dataset}'
        f' --dataset.root=/content/datasets/{dataset}'
        f' --dataset.image_transforms.enable={str(IMAGE_AUG).lower()}'
        f' --output_dir={output_root}/{run}'
        f' --job_name={run}'
        f' --batch_size={batch_size}'
        f' --steps={steps}'
        f' --save_freq={save_freq}'
        f' --log_freq={log_freq}'
        f' --num_workers={NUM_WORKERS}'
        ' --seed=1000'
        f' --wandb.enable={str(wandb).lower()}'
        f' --wandb.project={WANDB_PROJECT}'
        # Checkpoints already go to Drive; uploading each one as an artifact is slow.
        ' --wandb.disable_artifact=true'
    )
    print(cmd)
    get_ipython().system(cmd)

In [ ]:
# Throughput / memory test: 300 steps, nothing written to Drive. Run nvidia-smi in a
# terminal (or the next cell afterwards) to see the peak memory.
!rm -rf /content/smoke
train('img_smoke', 'so101_sim_pick_place_images_50', steps=300, save_freq=300,
      output_root='/content/smoke', wandb=False, log_freq=50)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

### 2.2 正式训练（291 回合）

吞吐测试正常后运行。run 名 `act_img_291`，与状态版的 `act_291_envnorm_retry` 用的是同样的 291 回合。

In [ ]:
train('act_img_291', 'so101_sim_pick_place_images_291')

### 2.3 480 回合（加了完成后保持、边界补充数据）

数据：正常示教 200（seed 2）+ 重抓 94（seed 3）+ |g| ∈ [30°, 45°] 的边界示教 100（seed 4）+ 边界 `midway` 重抓 86（seed 5）；每回合末尾有 2 s 停在初始姿态的保持帧。

帧数约是 291 回合的 1.9 倍，所以步数加到 100k（约 23 个 epoch，291 回合 60k 步约 27 个），每 10k 步存一个 checkpoint，闭环评估先看 60k 和 100k。耗时约是 291 回合 60k 的 1.7 倍。

In [ ]:
train('act_img_all_hold', 'so101_sim_pick_place_images_all_hold', steps=100_000)

## 3. 打包 checkpoint

评估只需要每个 checkpoint 的 `pretrained_model/` 目录。下载 zip 后在本机解压到工程的 `outputs/` 下。

In [ ]:
!cd {PROJECT} && zip -qr act_img_checkpoints.zip outputs/act_img_*/checkpoints/*/pretrained_model -x '*/last/*'
!ls -lh {PROJECT}/act_img_checkpoints.zip

## 断线续训

会话断开后，重新运行挂载、安装、W&B 登录、解压数据集和 `train` 定义的单元，再运行下面的单元。W&B 的 run id 保存在输出目录里，续训会接着原来的曲线记录。

In [ ]:
RUN = 'act_img_all_hold'   # or 'act_img_291'
!lerobot-train --resume=true --config_path={PROJECT}/outputs/{RUN}/checkpoints/last/pretrained_model/train_config.json